In [1]:
# Cell 1 - Setup
# pip install pdfplumber chromadb sentence-transformers pandas openpyxl

import re
import warnings
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
import pdfplumber
import chromadb
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
MANUALS_DIR = BASE_DIR / 'data' / 'manuals'
WORK_ORDERS_PATH = BASE_DIR / 'data' / 'synthetic_logs' / 'all_work_orders.csv'
ASSET_REGISTRY_PATH = BASE_DIR / 'data' / 'asset_registry.xlsx'
CHROMA_DIR = BASE_DIR / 'chroma_db'
CHROMA_COLLECTION_NAME = 'plantmind_knowledge'

try:
    asset_registry_df = pd.read_excel(ASSET_REGISTRY_PATH)
    print('Loaded asset registry:', len(asset_registry_df))
except Exception as e:
    print(f'Warning: could not load asset registry: {e}')
    asset_registry_df = pd.DataFrame()

print('Setup complete')

C:\Users\dev\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded asset registry: 23
Setup complete


In [5]:
# Cell 2 - PDF extraction (single file test)

def extract_pdf_text(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ''
                pages.append({'page_number': page_number, 'text': text})
    except Exception as e:
        print(f'Warning: failed to read {pdf_path}: {e}')
    return pages

sample_pdf = MANUALS_DIR / 'pump_centrifugal_manual.pdf'
sample_pages = extract_pdf_text(str(sample_pdf))

print(f'PDF: {sample_pdf.name}')
print(f'Total pages: {len(sample_pages)}')
if sample_pages:
    first_text = sample_pages[0]['text'] or ''
    print('First 500 characters:')
    print(first_text[:500])
else:
    print('No text extracted')

PDF: pump_centrifugal_manual.pdf
Total pages: 36
First 500 characters:
Manualslib.com - The Global Manuals Library
Manuals / Brands / Flowserve Manuals / Water Pump / FM Centrifugal Pump PCN=71576526 / User instructions / PDF
FLOWSERVE FM CENTRIFUGAL PUMP
PCN=71576526 USER INSTRUCTIONS
Table of Contents
Table of Contents
General
CE marking and approvals
Disclaimer
Copyright
Duty conditions
Safety
FM USER INSTRUCTIONS ENGLISH 71576526


In [6]:
# Cell 3 - PDF extraction (all files)

manual_lookup = {
    'pump_centrifugal_manual.pdf': 'Centrifugal Pump',
    'pump_submersible_manual.pdf': 'Submersible Pump',
    'motor_induction_manual.pdf': 'Induction Motor',
    'transformer_power_manual.pdf': 'Power Transformer',
    'panel_distribution_manual.pdf': 'Distribution Panel',
    'tower_cooling_manual.pdf': 'Cooling Tower',
    'chiller_centrifugal_manual.pdf': 'Centrifugal Chiller',
    'chiller_scroll_manual.pdf': 'Scroll Chiller',
    'compressor_air_manual.pdf': 'Air Compressor',
    'conveyor_belt_manual.pdf': 'Conveyor Belt',
    'dryer_refrigerated_manual.pdf': 'Refrigerated Dryer',
    'boiler_steam_manual.pdf': 'Steam Boiler',
}

manual_records = []
for pdf_file in sorted(MANUALS_DIR.glob('*.pdf')):
    asset_type = manual_lookup.get(pdf_file.name, 'Unknown')
    pages = extract_pdf_text(str(pdf_file))
    for page in pages:
        manual_records.append({
            'filename': pdf_file.name,
            'asset_type': asset_type,
            'page_number': page['page_number'],
            'text': page['text']
        })
    print(f'{pdf_file.name}: {len(pages)} pages extracted')

print(f'\nTotal manual page records: {len(manual_records)}')

boiler_steam_manual.pdf: 23 pages extracted
chiller_centrifugal_manual.pdf: 26 pages extracted
chiller_scroll_manual.pdf: 20 pages extracted
compressor_air_manual.pdf: 10 pages extracted
conveyor_belt_manual.pdf: 19 pages extracted
dryer_refrigerated_manual.pdf: 24 pages extracted
motor_induction_manual.pdf: 14 pages extracted
panel_distribution_manual.pdf: 16 pages extracted
pump_centrifugal_manual.pdf: 36 pages extracted
pump_submersible_manual.pdf: 14 pages extracted
tower_cooling_manual.pdf: 20 pages extracted
transformer_power_manual.pdf: 18 pages extracted

Total manual page records: 240


In [11]:
# Cell 4 - Chunking manuals

def clean_page_text(text: str) -> str:
    junk_lines = [
        "Downloaded from www.Manualslib.com manuals search engine",
        "Manualslib.com - The Global Manuals Library"
    ]
    for line in junk_lines:
        text = text.replace(line, "")
    return text.strip()

def chunk_text(text: str, chunk_size: int = 700, overlap: int = 100) -> List[str]:
    if not text:
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current = []
    current_len = 0
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        token_count = len(sentence.split())
        if current and current_len + token_count > chunk_size:
            chunk_text = ' '.join(current).strip()
            if chunk_text:
                chunks.append(chunk_text)
            overlap_text = current[-max(1, overlap // 120):]
            current = overlap_text.copy()
            current_len = sum(len(part.split()) for part in current)
        current.append(sentence)
        current_len += token_count
    if current:
        chunks.append(' '.join(current).strip())
    return chunks

manual_chunks = []
skipped_junk_count = 0

for record in manual_records:
    cleaned_text = clean_page_text(record['text'])
    chunks = chunk_text(cleaned_text)
    for chunk_text_value in chunks:
        if len(chunk_text_value.split()) < 30:
            skipped_junk_count += 1
            continue  # skip junk/near-empty chunks
        manual_chunks.append({
            'source_type': 'manual',
            'source_file': record['filename'],
            'asset_type': record['asset_type'],
            'asset_id': None,
            'related_asset_ids': None,
            'page_number': record['page_number'],
            'failure_mode': None,
            'text': chunk_text_value
        })

print(f'Total manual chunks: {len(manual_chunks)}')
print(f'Skipped junk/near-empty chunks: {skipped_junk_count}')

if manual_chunks:
    print('\nSample chunk 1:')
    print(manual_chunks[0]['text'][:500])
    print('Metadata:', manual_chunks[0])
    print('\nSample chunk 2:')
    print(manual_chunks[min(1, len(manual_chunks)-1)]['text'][:500])
    print('Metadata:', manual_chunks[min(1, len(manual_chunks)-1)])

Total manual chunks: 177
Skipped junk/near-empty chunks: 47

Sample chunk 1:
Manuals / Brands / York Manuals / Chiller / YZ / Engineering manual / PDF
YORK YZ ENGINEERING MANUAL
Table of Contents
Table of Contents
Nomenclature
Introduction
Unit Components
Equipment Overview
Codes and Standards
Chiller Options
Application Data
Unit Weights & Dimensions
Other ManualsLib Projects
Metadata: {'source_type': 'manual', 'source_file': 'chiller_centrifugal_manual.pdf', 'asset_type': 'Centrifugal Chiller', 'asset_id': None, 'related_asset_ids': None, 'page_number': 1, 'failure_mode': None, 'text': 'Manuals / Brands / York Manuals / Chiller / YZ / Engineering manual / PDF\nYORK YZ ENGINEERING MANUAL\nTable of Contents\nTable of Contents\nNomenclature\nIntroduction\nUnit Components\nEquipment Overview\nCodes and Standards\nChiller Options\nApplication Data\nUnit Weights & Dimensions\nOther ManualsLib Projects'}

Sample chunk 2:
YZ Engineering Guide
Form: 161.01-EG1 (0618)
Table of Contents
Nomencl

In [12]:
# Check chunks from later in the manual for real technical content
print("Total manual chunks:", len(manual_chunks))
print("\nSample from middle of dataset:")
mid_index = len(manual_chunks) // 2
print(manual_chunks[mid_index]['text'][:600])
print("Metadata:", manual_chunks[mid_index]['metadata'] if 'metadata' in manual_chunks[mid_index] else {k: manual_chunks[mid_index][k] for k in manual_chunks[mid_index] if k != 'text'})

print("\n\nSample from later in dataset:")
late_index = int(len(manual_chunks) * 0.75)
print(manual_chunks[late_index]['text'][:600])
print("Metadata:", {k: manual_chunks[late_index][k] for k in manual_chunks[late_index] if k != 'text'})

Total manual chunks: 177

Sample from middle of dataset:
Terminal box
Motor case
Terminal box mounting screw
(M4-4 pieces)
Tightening torque: Gasket
1.0 to 1.5 N·m (8.8 to 13.2 lb-in)
Note • Be sure to use the gasket attached. • Assemble so that foreign objects are not entered between the terminal box
and the motor case.
Metadata: {'source_type': 'manual', 'source_file': 'motor_induction_manual.pdf', 'asset_type': 'Induction Motor', 'asset_id': None, 'related_asset_ids': None, 'page_number': 11, 'failure_mode': None}


Sample from later in dataset:
FM USERINSTRUCTIONS ENGLISH 71576526-03/07
7 FAULTS;CAUSESAND REMEDIES
Insufficientflowrate
Irregularpumprunning
Driveroverloaded
Mechanicalsealleak
Equipmentvibration
Excessivepumpcasingtemperature
POSSIBLECAUSES SOLUTIONS
    Pumporsuctionpipenotcompletelyfilled -Checkandcompletefilling
    Airbubblesinpipes -Checkanddeaeratethepipes
   Suctionleveltoolow -Check:theavailableNPSH>therequiredNPSH
-Reducegeometricalsuctionlift
-Red

In [8]:
# Cell 5 - Loading and chunking work orders

try:
    work_orders_df = pd.read_csv(WORK_ORDERS_PATH)
    print('Loaded work orders:', len(work_orders_df))
except Exception as e:
    print(f'Warning: could not load work orders: {e}')
    work_orders_df = pd.DataFrame()

def build_work_order_text(row: pd.Series) -> str:
    return (
        f"Work Order {row.get('work_order_id', 'N/A')} for {row.get('asset_id', 'N/A')} ({row.get('asset_type', 'Unknown')}) on {row.get('date', 'Unknown')}: "
        f"technician {row.get('technician_name', 'Unknown')} reported {row.get('symptom_description', 'no symptoms')}. "
        f"Root cause: {row.get('root_cause', 'unknown')}. "
        f"Action taken: {row.get('action_taken', 'none')}. "
        f"Parts used: {row.get('parts_used', 'none')}. "
        f"Downtime: {row.get('downtime_hours', 'unknown')} hours."
    )

work_order_chunks = []
for _, row in work_orders_df.iterrows():
    try:
        text = build_work_order_text(row)
        related_ids = row.get('related_asset_ids', '')
        work_order_chunks.append({
            'source_type': 'work_order',
            'source_file': row.get('work_order_id', 'unknown'),
            'asset_type': row.get('asset_type', 'Unknown'),
            'asset_id': row.get('asset_id', None),
            'related_asset_ids': related_ids,
            'page_number': None,
            'failure_mode': row.get('failure_mode', None),
            'text': text
        })
    except Exception as e:
        print(f'Warning: skipped a work order row due to error: {e}')

print(f'Total work order chunks: {len(work_order_chunks)}')
if work_order_chunks:
    print('Sample chunk 1:')
    print(work_order_chunks[0]['text'][:500])
    print('Metadata:', work_order_chunks[0])
    print('\nSample chunk 2:')
    print(work_order_chunks[min(1, len(work_order_chunks)-1)]['text'][:500])
    print('Metadata:', work_order_chunks[min(1, len(work_order_chunks)-1)])

Loaded work orders: 388
Total work order chunks: 388
Sample chunk 1:
Work Order WO-0001 for DRYER-202 (Refrigerated Air Dryer) on 2023-01-05: technician Jameson Thorne reported Compressor short cycling (`18 Hz`) on low-pressure cutout switch (`< 31 PSI`). Tech observed accelerated oil residue on copper brazed joint near evaporator.. Root cause: Degraded Schrader access valve core O-ring (`Spec-854`) allowed slow loss of R-134a/R-407C refrigerant charge (`-34%` loss) over past 6 months.. Action taken: Repaired pinhole braze leak at evaporator inlet header, install
Metadata: {'source_type': 'work_order', 'source_file': 'WO-0001', 'asset_type': 'Refrigerated Air Dryer', 'asset_id': 'DRYER-202', 'related_asset_ids': nan, 'page_number': None, 'failure_mode': 'refrigerant leak', 'text': 'Work Order WO-0001 for DRYER-202 (Refrigerated Air Dryer) on 2023-01-05: technician Jameson Thorne reported Compressor short cycling (`18 Hz`) on low-pressure cutout switch (`< 31 PSI`). Tech observed accele

In [13]:
# Cell 6 - Combine all chunks

all_chunks = []
for chunk in manual_chunks:
    all_chunks.append(chunk)
for chunk in work_order_chunks:
    all_chunks.append(chunk)

source_breakdown = {}
for chunk in all_chunks:
    source_breakdown[chunk['source_type']] = source_breakdown.get(chunk['source_type'], 0) + 1

print(f'Total combined chunks: {len(all_chunks)}')
print('Breakdown by source_type:', source_breakdown)

Total combined chunks: 565
Breakdown by source_type: {'manual': 177, 'work_order': 388}


In [14]:
# Cell 7 - Embedding and storing in Chroma
import math

def clean_meta_value(v):
    if v is None:
        return ""
    if isinstance(v, float) and math.isnan(v):
        return ""
    return v
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection(name=CHROMA_COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(name=CHROMA_COLLECTION_NAME)

model = SentenceTransformer('all-MiniLM-L6-v2')

batch_size = 50
for index in range(0, len(all_chunks), batch_size):
    batch = all_chunks[index:index + batch_size]
    texts = [item['text'] for item in batch]
    embeddings = model.encode(texts, convert_to_numpy=True).tolist()
    ids = [f"chunk_{index + i}" for i in range(len(batch))]
    metadata = [{
    'source_type': clean_meta_value(item['source_type']),
    'source_file': clean_meta_value(item['source_file']),
    'asset_type': clean_meta_value(item['asset_type']),
    'asset_id': clean_meta_value(item['asset_id']),
    'related_asset_ids': clean_meta_value(item['related_asset_ids']),
    'page_number': clean_meta_value(item['page_number']),
    'failure_mode': clean_meta_value(item['failure_mode'])
    } for item in batch]
    collection.add(ids=ids, documents=texts, metadatas=metadata)
    if index % 100 == 0:
        print(f'Added {index + len(batch)} chunks to Chroma...')

print(f'Total chunks stored in collection: {collection.count()}')

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 167.52it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Added 50 chunks to Chroma...
Added 150 chunks to Chroma...
Added 250 chunks to Chroma...
Added 350 chunks to Chroma...
Added 450 chunks to Chroma...
Added 550 chunks to Chroma...
Total chunks stored in collection: 565


In [15]:
# Cell 8 - Test retrieval

queries = [
    'What causes cavitation in centrifugal pumps?',
    'What maintenance has PUMP-101 needed?',
    'Is there a connection between motor bearing failures and the pumps they drive?'
]

for query in queries:
    print(f'\nQuery: {query}')
    try:
        query_embedding = model.encode([query], convert_to_numpy=True).tolist()[0]
        results = collection.query(query_embeddings=[query_embedding], n_results=5)
        for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0]), start=1):
            text = (doc or '')[:200]
            print(f'{i}. {text}...')
            print('   Metadata:', meta)
    except Exception as e:
        print(f'Warning: retrieval failed for this query: {e}')


Query: What causes cavitation in centrifugal pumps?
1. Work Order WO-0022 for PUMP-101 (Centrifugal Pump) on 2023-02-09: technician Hannah Abbott reported Severe cavitation noise (`grinding`) detected across pump volute with suction pressure dropping by `...
   Metadata: {'page_number': '', 'asset_id': 'PUMP-101', 'asset_type': 'Centrifugal Pump', 'source_file': 'WO-0022', 'related_asset_ids': '', 'failure_mode': 'cavitation', 'source_type': 'work_order'}
2. Work Order WO-0387 for PUMP-101 (Centrifugal Pump) on 2024-12-30: technician Marcus Vance reported Ops noted pump running smoothly but failing to meet process target gpm (`29%` deficit). Hydraulic per...
   Metadata: {'source_type': 'work_order', 'asset_id': 'PUMP-101', 'source_file': 'WO-0387', 'page_number': '', 'related_asset_ids': 'MOTOR-101', 'failure_mode': 'impeller wear', 'asset_type': 'Centrifugal Pump'}
3. Work Order WO-0186 for PUMP-101 (Centrifugal Pump) on 2023-12-20: technician Sarah Jenkins reported Ops noted pump r

In [17]:
# Cell 9 - Hybrid retrieval function (manual + work_order merge)

def hybrid_retrieve(query: str, n_manual: int = 5, n_workorder: int = 3):
    manual_results = collection.query(
        query_texts=[query],
        n_results=n_manual,
        where={"source_type": "manual"}
    )
    workorder_results = collection.query(
        query_texts=[query],
        n_results=n_workorder,
        where={"source_type": "work_order"}
    )
    combined_context = []
    for doc, meta in zip(manual_results['documents'][0], manual_results['metadatas'][0]):
        combined_context.append({"text": doc, "metadata": meta})
    for doc, meta in zip(workorder_results['documents'][0], workorder_results['metadatas'][0]):
        combined_context.append({"text": doc, "metadata": meta})
    return combined_context

# Test it
test_result = hybrid_retrieve("What causes cavitation in centrifugal pumps?")
for i, item in enumerate(test_result, 1):
    print(f"{i}. [{item['metadata']['source_type']}] {item['text'][:200]}")
    print(f"   Metadata: {item['metadata']}\n")

1. [manual] Please read these operating instructions carefully and observe the
notes given. Use these operating instructions to familiarise your-
self with your Pump, its proper use and the notes on safety. For s
   Metadata: {'asset_type': 'Submersible Pump', 'related_asset_ids': '', 'page_number': 3, 'failure_mode': '', 'source_file': 'pump_submersible_manual.pdf', 'source_type': 'manual', 'asset_id': ''}

2. [manual] Electrical safety: Operating instructions:
➜ Before operating the pump look to see if
there is any damage to the pump (especially
DANGER ! Electric shock ! to the power cable and plug). The pump must 
   Metadata: {'related_asset_ids': '', 'page_number': 4, 'source_file': 'pump_submersible_manual.pdf', 'asset_type': 'Submersible Pump', 'source_type': 'manual', 'failure_mode': '', 'asset_id': ''}

3. [manual] 16
BG
9. Warranty
GARDENA guarantees this product for 2 years from date of
purchase. This guarantee covers all serious defects of the unit
that can be proved to be 